In [9]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session
from google.cloud import storage

# ==================================================
# 1. CONFIGURATION
# ==================================================

PROJECT_ID = "fintech-lakehouse-dev"
REGION = "us-central1"

CATALOG_NAME = "fintech_lakehouse"
CATALOG_ID = "fintech_lakehouse_bucket"

REST_URI = "https://biglake.googleapis.com/iceberg/v1/restcatalog"

WAREHOUSE = (
    f"bl://projects/{PROJECT_ID}/catalogs/{CATALOG_ID}"
)

# ==================================================
# 2. DATAPROC RUNTIME CONFIGURATION
# ==================================================

session_config = Session()

# Spark 4.x runtime
session_config.runtime_config.version = "3.0"

## JVM DEPENDENCY JARS
BUCKET = "fintech_lakehouse_bucket"
core = [
    "spark-sql-kafka-0-10_2.13-4.0.2.jar",
    "spark-token-provider-kafka-0-10_2.13-4.0.2.jar",
    "kafka-clients-3.9.0.jar",
    "commons-pool2-2.12.0.jar",
    "spark-avro_2.13-4.0.2.jar",
]
auth = [b.name for b in storage.Client().list_blobs(BUCKET, prefix="jars/auth/") if b.name.endswith(".jar")]

jars = [f"gs://{BUCKET}/jars/{j}" for j in core] + [f"gs://{BUCKET}/{n}" for n in auth]
session_config.runtime_config.properties["spark.jars"] = ",".join(jars)
print(len(jars), "jars")

# ==================================================
# 3. ICEBERG CATALOG CONFIGURATION
# ==================================================

builder = (
    DataprocSparkSession.builder
    .appName("FintechLakehouse")
    .dataprocSessionConfig(session_config)

    .config("spark.sql.defaultCatalog", CATALOG_NAME)

    # Iceberg REST catalog
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.type",
        "rest"
    )
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.uri",
        REST_URI
    )
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.warehouse",
        WAREHOUSE
    )

    # GCP project
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.header.x-goog-user-project",
        PROJECT_ID
    )

    # Google authentication
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.rest.auth.type",
        "org.apache.iceberg.gcp.auth.GoogleAuthManager"
    )

    # GCS FileIO
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.io-impl",
        "org.apache.iceberg.gcp.gcs.GCSFileIO"
    )

    # Iceberg SQL extensions
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )

    # Credential vending
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.header.X-Iceberg-Access-Delegation",
        "vended-credentials"
    )
)

# ==================================================
# 4. CREATE SPARK SESSION
# ==================================================

spark = builder.getOrCreate()

print("Spark session initialized!")
print("Spark version:", spark.version)

41 jars
Using existing Dataproc Session (configuration changes may not be applied): https://console.cloud.google.com/dataproc/interactive/us-central1/sc-20261109-150634-02u6ij?project=fintech-lakehouse-dev


Spark session initialized!
Spark version: 4.0.2


In [10]:
from pyspark.sql import functions as F
from pyspark.sql.avro.functions import from_avro
from pyspark.sql.functions import col, from_unixtime

PROJECT_ID = "fintech-lakehouse-dev"

TOPIC_NAME = "card-transactions-topic"

CATALOG = 'fintech_lakehouse'

BRONZE_SCHEMA = f"{CATALOG}.bronze"

BRONZE1 = f"{BRONZE_SCHEMA}.raw_transactions"

SILVER_SCHEMA = f"{CATALOG}.silver"

SILVER1 = f"{SILVER_SCHEMA}.decoded_transactions"

BUCKET = "fintech_lakehouse_bucket"

CHECKPOINT2 = f"gs://{BUCKET}/checkpoints/silver/decoded_txns"

print (SILVER1)

fintech_lakehouse.silver.decoded_transactions


TABLE CREATION

In [11]:
spark.sql("""
    CREATE NAMESPACE IF NOT EXISTS fintech_lakehouse.gold
""")

DataFrame[]

In [12]:
spark.sql("""
    SHOW NAMESPACES IN fintech_lakehouse
""").show()

+---------+
|namespace|
+---------+
|   bronze|
|     gold|
|   silver|
+---------+



In [13]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS fintech_lakehouse.gold.customer_txn_features_5min (
  customer_id STRING, currency STRING,
  bucket_start TIMESTAMP, bucket_end TIMESTAMP,
  txn_count BIGINT, amount_sum DECIMAL(18,2), amount_max DECIMAL(18,2),
  intl_txn_count BIGINT, high_risk_txn_count BIGINT, max_fraud_score DOUBLE,
  merchants ARRAY<STRING>, countries ARRAY<STRING>, devices ARRAY<STRING>,
  last_event_ts TIMESTAMP, updated_at TIMESTAMP
) USING iceberg
PARTITIONED BY (days(bucket_start))""")

DataFrame[]

MAIN

In [14]:
from pyspark.sql import DataFrame, SparkSession, functions as F

HIGH_RISK = 0.7            # keep identical to the batch gold's high_risk_threshold
WATERMARK = "10 minutes"   # how late an event may arrive and still be counted by the stream
BUCKET = "5 minutes"       # tumbling bucket size; rolling windows are summed from buckets at read time

FEATURE_TABLE = f"{CATALOG}.gold.customer_txn_features_5min"
SILVER_TABLE = f"{CATALOG}.silver.decoded_transactions"
checkpoint_loc ="gs://fintech_lakehouse_bucket/gold/checkpoint/customer_txn_features_5min"

In [15]:
def bucket_features(txns: DataFrame) -> DataFrame:
    """One definition of the features, used by the streaming job AND by batch backfill (no train/serve skew).
    Input columns: transaction_id, customer_id, event_timestamp (epoch ms), amount, currency,
    merchant_name, location_country, is_international, device_fingerprint, fraud_risk_score."""
    t = (
        txns.withColumn("event_ts", (F.col("event_timestamp") / 1000).cast("timestamp"))
            .withWatermark("event_ts", WATERMARK)
    )
    # Kafka is at-least-once: dedupe. Streaming needs the watermark-bounded variant (bounded state);
    # that variant is not allowed on batch DataFrames, which can use a plain dropDuplicates.
    t = t.dropDuplicatesWithinWatermark(["transaction_id"]) if txns.isStreaming else t.dropDuplicates(["transaction_id"])
    t = t.withColumn("amount", F.col("amount").cast("decimal(18,2)"))
    return (
        t.groupBy("customer_id", F.upper("currency").alias("currency"), F.window("event_ts", BUCKET).alias("w"))
         .agg(
             F.count("*").alias("txn_count"),
             F.sum("amount").cast("decimal(18,2)").alias("amount_sum"),
             F.max("amount").cast("decimal(18,2)").alias("amount_max"),
             F.sum(F.col("is_international").cast("int")).alias("intl_txn_count"),
             F.sum((F.col("fraud_risk_score") >= HIGH_RISK).cast("int")).alias("high_risk_txn_count"),
             F.max("fraud_risk_score").alias("max_fraud_score"),
             F.collect_set("merchant_name").alias("merchants"),
             F.collect_set("location_country").alias("countries"),
             F.collect_set("device_fingerprint").alias("devices"),
             F.max("event_ts").alias("last_event_ts"),
         )
         .select(
             "customer_id", "currency",
             F.col("w.start").alias("bucket_start"), F.col("w.end").alias("bucket_end"),
             "txn_count", "amount_sum", "amount_max", "intl_txn_count", "high_risk_txn_count",
             "max_fraud_score", "merchants", "countries", "devices", "last_event_ts",
             F.current_timestamp().alias("updated_at"),
         )
    )


def upsert_iceberg(batch_df: DataFrame, batch_id: int) -> None:
    """Update-mode batches carry the full current value of each changed bucket, so MERGE is idempotent."""
    batch_df.createOrReplaceTempView("updates")
    batch_df.sparkSession.sql(f"""
        MERGE INTO {FEATURE_TABLE} t
        USING updates s
        ON  t.customer_id = s.customer_id AND t.currency = s.currency AND t.bucket_start = s.bucket_start
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)


def silver_stream(spark: SparkSession, lookback_hours: int = 2) -> DataFrame:
    import time
    start_ms = str(int((time.time() - lookback_hours * 3600) * 1000))
    return (spark.readStream.format("iceberg")
            .option("stream-from-timestamp", start_ms)          # don't replay the whole table
            .load(SILVER_TABLE))


def start_stream(source_df: DataFrame, upsert_fn, checkpoint: str, trigger: dict):
    """trigger examples: {"processingTime": "1 minute"} (continuous) or {"availableNow": True} (cheap POC runs)."""
    return (bucket_features(source_df).writeStream
            .outputMode("update")
            .foreachBatch(upsert_fn)
            .option("checkpointLocation", checkpoint)
            .trigger(**trigger)
            .start())


def backfill(spark: SparkSession, start_date: str, end_date: str) -> None:
    """Batch recompute of buckets for [start_date, end_date] from silver, fixing anything the stream dropped
    as too late. Same bucket_features(), run as a normal batch query."""
    silver = spark.table(SILVER_TABLE).where(
        (F.col("event_time_utc") >= F.lit(start_date).cast("timestamp"))
        & (F.col("event_time_utc") < F.date_add(F.lit(end_date).cast("date"), 1).cast("timestamp"))
    )
    upsert_iceberg(bucket_features(silver), batch_id=-1)


# ----------------------------------------------------------------------------- read side (serving)
def rolling_features(buckets: list, now, windows=(("1h", 3600), ("24h", 86400))) -> dict:
    """Sum 5-minute bucket rows (dicts, ONE customer + ONE currency) into rolling-window features.
    Window edges snap to bucket boundaries, so edge error is at most one bucket (5 min).
    `now` and bucket_start must be timezone-naive UTC datetimes."""
    from datetime import datetime, timedelta
    bucket_s = 300
    secs_since_epoch = (now - datetime(1970, 1, 1)).total_seconds()
    now_floor = now - timedelta(seconds=secs_since_epoch % bucket_s)
    out = {}
    for label, secs in windows:
        cutoff = now_floor - timedelta(seconds=secs)
        rows = [b for b in buckets if cutoff <= b["bucket_start"] <= now]
        out[f"txn_count_{label}"] = sum(b["txn_count"] for b in rows)
        out[f"amount_sum_{label}"] = sum(b["amount_sum"] for b in rows)
        out[f"intl_txn_count_{label}"] = sum(b["intl_txn_count"] for b in rows)
        out[f"high_risk_txn_count_{label}"] = sum(b["high_risk_txn_count"] for b in rows)
        out[f"distinct_merchants_{label}"] = len({m for b in rows for m in b["merchants"]})
        out[f"distinct_countries_{label}"] = len({c for b in rows for c in b["countries"]})
        out[f"distinct_devices_{label}"] = len({d for b in rows for d in b["devices"]})
    out["seconds_since_last_txn"] = (
        int((now - max(b["last_event_ts"] for b in buckets)).total_seconds()) if buckets else None
    )
    return out

In [16]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

# POC mode: process whatever is new, then stop (no 24/7 cost)
q = start_stream(silver_stream(spark), upsert_iceberg,
                 checkpoint=checkpoint_loc,
                 trigger={"availableNow": True})
q.awaitTermination()

# continuous mode, for demos: swap the trigger for {"processingTime": "1 minute"}

# nightly repair of late or dropped events
backfill(spark, "2026-11-08", "2026-11-09")

KeyboardInterrupt: 